In [128]:
pip install sentence-transformers faiss-cpu


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Import libraries and load the chunks

In [129]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

In [130]:
# Load the chunk dataset

chunks_df = pd.read_csv(
    "../data/guvi_chunks.csv"
)

print("Dataset Shape:", chunks_df.shape)
chunks_df.head()

Dataset Shape: (518, 6)


,url,title,category,chunk_id,chunk_text,word_count
0,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,0,Best Jira: Jira project management Course Onli...,200
1,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,1,& Machine Learning New Intel & IIT-M Pravartak...,200
2,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,2,"tech learning is easy, fun, and curated specia...",200
3,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,3,Now > WebKata: An interactive platform to mast...,200
4,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,4,or code and unlock exciting rewards—Amazon vou...,200


### Load the embedding model

In [131]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8515.93it/s]


Embedding model loaded successfully!


### Create embeddings for all chunks

In [132]:
embeddings = model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 17/17 [00:01<00:00,  8.92it/s]


In [133]:
print("Embedding Shape:", embeddings.shape)

Embedding Shape: (518, 384)


### Normalize the embeddings

In [134]:
embeddings = embeddings.astype("float32")

faiss.normalize_L2(embeddings)

In [135]:
print("Embeddings ready for FAISS:", embeddings.shape)

Embeddings ready for FAISS: (518, 384)


### Create the FAISS index

In [136]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created successfully!")
print("Total vectors in index:", index.ntotal)

FAISS index created successfully!
Total vectors in index: 518


### Test semantic search

In [137]:
def search_guvi(query, top_k=5):
    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = chunks_df.iloc[
        indices[0]
    ].copy()

    results["score"] = scores[0]

    return results[
        [
            "title",
            "category",
            "chunk_text",
            "score"
        ]
    ]

In [138]:
results = search_guvi(
    "What courses are available for data science?"
)

results

,title,category,chunk_text,score
407,HCL GUVI - Zen Class Success Stories,Live Program,such a talented team of instructors. The progr...,0.589423
410,HCL GUVI - Zen Class Success Stories,Live Program,"journey has been immensely fulfilling, and I’m...",0.585288
411,HCL GUVI - Zen Class Success Stories,Live Program,skills in real-world projects and exploring ne...,0.564904
413,HCL GUVI - Zen Class Success Stories,Live Program,the time and effort you put into helping me gr...,0.538263
141,HCL GUVI - Free Resources,Resource,Download Now Master the Art of Data Science - ...,0.530491


In [139]:
# Improve the search by including title + category in embeddings

chunks_df["embedding_text"] = (
    chunks_df["title"].fillna("") + " " +
    chunks_df["category"].fillna("") + " " +
    chunks_df["chunk_text"].fillna("")
)

In [140]:
# regenerate embeddings:
embeddings = model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True
).astype("float32")

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches: 100%|██████████| 17/17 [00:01<00:00,  9.68it/s]


In [141]:
# Normalize

faiss.normalize_L2(embeddings)

In [142]:
# Rebuild the index

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 518


In [143]:
results = search_guvi(
    "What courses are available for data science?"
)

results

,title,category,chunk_text,score
410,HCL GUVI - Zen Class Success Stories,Live Program,"journey has been immensely fulfilling, and I’m...",0.559033
407,HCL GUVI - Zen Class Success Stories,Live Program,such a talented team of instructors. The progr...,0.541541
492,Data Science Course With Placement | Zen Class...,Live Program,Industry Student Graduated and Looking for a C...,0.511687
494,Data Science Course With Placement | Zen Class...,Live Program,urbanrise integra Xerago Xerago TIMS Ramco AI ...,0.506662
411,HCL GUVI - Zen Class Success Stories,Live Program,skills in real-world projects and exploring ne...,0.504877


The second search is still not good enough. The important point is that changing the embedding text did not solve the problem.

In [144]:
# Check whether Data Science exists in our knowledge base

data_science_rows = chunks_df[
    chunks_df["embedding_text"]
    .str.contains(
        "data science",
        case=False,
        na=False
    )
]

print(
    "Chunks containing 'data science':",
    len(data_science_rows)
)

Chunks containing 'data science': 205


In [145]:
# inspect the pages

data_science_rows[
    [
        "title",
        "category",
        "url"
    ]
].drop_duplicates()

,title,category,url
0,Best Jira: Jira project management Course Onli...,Course,https://www.guvi.in/courses/project-management...
10,HCL GUVI | Learn to code in your native language,Other,https://www.guvi.in
26,Referral | HCL GUVI,Other,https://www.guvi.in/referral
32,HCL GUVI | Learn to code in your native language,Practice,https://www.guvi.in/code-kata
41,Zen Class - Career Programs from HCL GUVI,Live Program,https://www.guvi.in/zen-class
68,Rewards | HCL GUVI,Other,https://www.guvi.in/rewards
72,Best Claude Code Course Online with Certificat...,Course,https://www.guvi.in/courses/machine-learning-a...
82,"Learn Anytime, Anywhere with the HCL GUVI App",Other,https://app.guvi.in
89,Leader Board | HCL GUVI,Other,https://www.guvi.in/leader-board
94,HCL GUVI | courses,Course,https://www.guvi.in/courses/tamil/programming/...


In [146]:
# Check titles specifically for Data Science

data_science_titles = chunks_df[
    chunks_df["title"]
    .str.contains(
        "data science",
        case=False,
        na=False
    )
][
    ["title", "category", "url"]
].drop_duplicates()

print("Data Science pages found:", len(data_science_titles))

data_science_titles

Data Science pages found: 1


,title,category,url
486,Data Science Course With Placement | Zen Class...,Live Program,https://www.guvi.in/zen-class/data-science-cou...


In [147]:
chunks_df = pd.read_csv(
    "../data/guvi_chunks.csv"
)

print("Updated Chunk Shape:", chunks_df.shape)

Updated Chunk Shape: (518, 6)


In [148]:
chunks_df["embedding_text"] = (
    chunks_df["title"].fillna("") + " " +
    chunks_df["category"].fillna("") + " " +
    chunks_df["chunk_text"].fillna("")
)

In [149]:
print(chunks_df.columns)

Index(['url', 'title', 'category', 'chunk_id', 'chunk_text', 'word_count',
       'embedding_text'],
      dtype='object')


In [150]:
embeddings = model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True
).astype("float32")

Batches: 100%|██████████| 17/17 [00:01<00:00, 10.19it/s]


In [151]:
faiss.normalize_L2(embeddings)

In [152]:
print("Updated Embedding Shape:", embeddings.shape)

Updated Embedding Shape: (518, 384)


In [153]:
# Rebuild the FAISS index with the updated embeddings

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index rebuilt successfully!")
print("Total vectors in index:", index.ntotal)

FAISS index rebuilt successfully!
Total vectors in index: 518


In [154]:
# Test the Data Science query again

In [155]:
results = search_guvi(
    "What courses are available for data science?"
)

results

,title,category,chunk_text,score
410,HCL GUVI - Zen Class Success Stories,Live Program,"journey has been immensely fulfilling, and I’m...",0.559033
407,HCL GUVI - Zen Class Success Stories,Live Program,such a talented team of instructors. The progr...,0.541541
492,Data Science Course With Placement | Zen Class...,Live Program,Industry Student Graduated and Looking for a C...,0.511687
494,Data Science Course With Placement | Zen Class...,Live Program,urbanrise integra Xerago Xerago TIMS Ramco AI ...,0.506662
411,HCL GUVI - Zen Class Success Stories,Live Program,skills in real-world projects and exploring ne...,0.504877


In [156]:
# Test title-aware filtering

data_science_df = chunks_df[
    chunks_df["title"].str.contains(
        "data science",
        case=False,
        na=False
    )
].copy()

print("Data Science chunks:", len(data_science_df))

Data Science chunks: 27


In [157]:
# Build a temporary FAISS index only for Data Science chunks

data_science_embeddings = model.encode(
    data_science_df["embedding_text"].tolist()
).astype("float32")

faiss.normalize_L2(data_science_embeddings)

In [158]:
ds_index = faiss.IndexFlatIP(
    data_science_embeddings.shape[1]
)

ds_index.add(data_science_embeddings)

In [159]:
print(
    "Vectors in Data Science index:",
    ds_index.ntotal
)

Vectors in Data Science index: 27


In [160]:
# Search only inside the Data Science chunks

query = "What courses are available for data science?"

query_embedding = model.encode(
    [query]
).astype("float32")

faiss.normalize_L2(query_embedding)

scores, indices = ds_index.search(
    query_embedding,
    5
)

ds_results = data_science_df.iloc[
    indices[0]
].copy()

ds_results["score"] = scores[0]

ds_results[
    [
        "title",
        "category",
        "chunk_text",
        "score"
    ]
]

,title,category,chunk_text,score
492,Data Science Course With Placement | Zen Class...,Live Program,Industry Student Graduated and Looking for a C...,0.511687
494,Data Science Course With Placement | Zen Class...,Live Program,urbanrise integra Xerago Xerago TIMS Ramco AI ...,0.506662
500,Data Science Course With Placement | Zen Class...,Live Program,"expert-led sessions, self-paced modules, mento...",0.494666
493,Data Science Course With Placement | Zen Class...,Live Program,"analytics, intelligent automation, recommendat...",0.481755
486,Data Science Course With Placement | Zen Class...,Live Program,Data Science Course With Placement Zen Class H...,0.442144


In [161]:
# Create an improved search function

def improved_search(query, top_k=5, candidate_k=15):
    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        candidate_k
    )

    results = chunks_df.iloc[
        indices[0]
    ].copy()

    results["score"] = scores[0]

    # Give a small bonus when query words appear in the page title
    query_words = {
        word.lower()
        for word in query.split()
        if len(word) > 3
    }

    def title_bonus(title):
        title = str(title).lower()

        matches = sum(
            word in title
            for word in query_words
        )

        return matches * 0.05

    results["final_score"] = (
        results["score"] +
        results["title"].apply(title_bonus)
    )

    results = results.sort_values(
        "final_score",
        ascending=False
    ).head(top_k)

    return results[
        [
            "title",
            "category",
            "chunk_text",
            "score",
            "final_score"
        ]
    ]

In [162]:
results = improved_search(
    "What courses are available for data science?"
)

results

,title,category,chunk_text,score,final_score
492,Data Science Course With Placement | Zen Class...,Live Program,Industry Student Graduated and Looking for a C...,0.511687,0.561687
410,HCL GUVI - Zen Class Success Stories,Live Program,"journey has been immensely fulfilling, and I’m...",0.559033,0.559033
494,Data Science Course With Placement | Zen Class...,Live Program,urbanrise integra Xerago Xerago TIMS Ramco AI ...,0.506662,0.556662
500,Data Science Course With Placement | Zen Class...,Live Program,"expert-led sessions, self-paced modules, mento...",0.494666,0.544666
407,HCL GUVI - Zen Class Success Stories,Live Program,such a talented team of instructors. The progr...,0.541541,0.541541


In [163]:
# Save the FAISS index

faiss.write_index(
    index,
    "../data/guvi_faiss.index"
)

print("FAISS index saved successfully!")

FAISS index saved successfully!


In [164]:
# Save the chunk metadata too

chunks_df.to_csv(
    "../data/guvi_chunks_with_embeddings_text.csv",
    index=False
)

print("Chunk metadata saved successfully!")

Chunk metadata saved successfully!


In [165]:
loaded_index = faiss.read_index(
    "../data/guvi_faiss.index"
)

print("FAISS index loaded successfully!")
print("Total vectors:", loaded_index.ntotal)
print("Vector dimension:", loaded_index.d)

FAISS index loaded successfully!
Total vectors: 518
Vector dimension: 384


In [166]:
print("Updated Chunk Shape:", chunks_df.shape)
print("Updated Embedding Shape:", embeddings.shape)
print("Total vectors in index:", index.ntotal)

Updated Chunk Shape: (518, 7)
Updated Embedding Shape: (518, 384)
Total vectors in index: 518


In [167]:
test_query = "What courses are available in GUVI?"

results = improved_search(
    test_query,
    top_k=10,
    candidate_k=30
)

print("Query:", test_query)
print()

for i, row in results.iterrows():
    print("Title:", row["title"])
    print("Category:", row["category"])
    print("Score:", round(row["final_score"], 4))
    print("Content:", row["chunk_text"][:300])
    print()
    print("-" * 80)
    print()

Query: What courses are available in GUVI?

Title: Explore All GUVI Courses | Learn Coding in English, Hindi, Tamil & More
Category: Course
Score: 0.7415
Content: unique link or code and unlock exciting rewards—Amazon vouchers, iPhones, and more. A Win-Win. Explore More Profile Your HCL GUVI profile is your digital portfolio! Track progress, showcase skills, add projects, and build a resume. Keep it updated—opportunities await! Explore More That's It! You Are

--------------------------------------------------------------------------------

Title: Explore All GUVI Courses | Learn Coding in English, Hindi, Tamil & More
Category: Course
Score: 0.6826
Content: Vernacular Imprint—where tech learning is easy, fun, and curated specially for you. Incubated by IIT Madras & IIM Ahmedabad in 2014 and now part of HCL Group, we're making quality tech education accessible to all. Join 3M+ learners breaking barriers and upskilling for a brighter future. We're here t

--------------------------------

In [168]:
course_check = chunks_df[
    chunks_df["title"].str.contains(
        "GUVI Courses|HCL GUVI \\| courses",
        case=False,
        na=False,
        regex=True
    )
]

print("Total Matching Chunks:", len(course_check))
print()

print(
    course_check[
        ["title", "category"]
    ].value_counts()
)

Total Matching Chunks: 27

title                                                                    category
HCL GUVI | courses                                                       Course      22
Explore All GUVI Courses | Learn Coding in English, Hindi, Tamil & More  Course       5
Name: count, dtype: int64


In [169]:
print("Updated Chunk Shape:", chunks_df.shape)
print("Updated Embedding Shape:", embeddings.shape)
print("Total vectors in index:", index.ntotal)

Updated Chunk Shape: (518, 7)
Updated Embedding Shape: (518, 384)
Total vectors in index: 518


In [170]:
test_query = "What courses are available in GUVI?"

results = improved_search(
    test_query,
    top_k=10,
    candidate_k=30
)

print("Query:", test_query)
print()

for _, row in results.iterrows():
    print("Title:", row["title"])
    print("Category:", row["category"])
    print("Score:", round(row["final_score"], 4))
    print("Content:", row["chunk_text"][:300])
    print()
    print("-" * 80)
    print()

Query: What courses are available in GUVI?

Title: Explore All GUVI Courses | Learn Coding in English, Hindi, Tamil & More
Category: Course
Score: 0.7415
Content: unique link or code and unlock exciting rewards—Amazon vouchers, iPhones, and more. A Win-Win. Explore More Profile Your HCL GUVI profile is your digital portfolio! Track progress, showcase skills, add projects, and build a resume. Keep it updated—opportunities await! Explore More That's It! You Are

--------------------------------------------------------------------------------

Title: Explore All GUVI Courses | Learn Coding in English, Hindi, Tamil & More
Category: Course
Score: 0.6826
Content: Vernacular Imprint—where tech learning is easy, fun, and curated specially for you. Incubated by IIT Madras & IIM Ahmedabad in 2014 and now part of HCL Group, we're making quality tech education accessible to all. Join 3M+ learners breaking barriers and upskilling for a brighter future. We're here t

--------------------------------